# 06 · Two-stage cache (decode once, extract fast)

Reading `.log` text is the bottleneck at scale (~0.3 s/log, I/O-bound on the
external drive). This splits the work:

1. **Stage 1 - decode** each `.log` into a compact `.npz` (4 sensors + 2 wheel
   detectors, float32 ~0.7 MB vs ~12 MB text). Reads the text **once**.
2. **Stage 2 - extract features** from the `.npz` cache (~16x faster reads).
   Re-tuning feature config (`cp_decimate`, frequency range, QC) re-runs only
   stage 2 and never re-reads the text.

Put the cache on your **internal SSD** (the project folder is already on it) so
you don't copy the raw text anywhere - stage 2 then reads from SSD.

**Production note:** the serving app receives only `.log` files and calls
`extract_features(path)` directly - the cache is a training-time optimization.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config
DATA_ROOT = r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts"
W10  = os.path.join(DATA_ROOT, "W10v140")
BLADE = os.path.join(DATA_ROOT, "Bladenr8252")
RAW_CACHE = "raw_cache"      # decoded .npz (keep on internal SSD)
STORE = "feature_store"      # extracted features

cfg = Config()
cfg.cp_decimate = 4          # verify vs 1 on a sample before trusting


## Stage 1 - decode .log -> .npz (one-time; reads text once)
Resumable: re-running skips already-decoded logs. `LIMIT=None` for all.


In [ ]:
from intelliswitch.io import build_raw_cache
LIMIT = 500
build_raw_cache(W10, RAW_CACHE, cfg, n_workers=os.cpu_count(), limit=LIMIT)


## Stage 2 - extract features from the cache (fast, re-runnable)
Reads the `.npz` cache. Changing feature config rebuilds this stage only.


In [ ]:
from intelliswitch.pipeline import build_feature_store, load_feature_store, store_summary
build_feature_store(W10, STORE, cfg,
                    n_workers=os.cpu_count(),
                    shard_size=400,
                    limit=LIMIT,
                    raw_cache_dir=RAW_CACHE)   # <- read from stage-1 cache
store_summary(STORE)


## Attach labels + train (instant off the cached features)


In [ ]:
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels
from intelliswitch.model import fit_per_location

deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)
data = attach_labels(load_feature_store(STORE), deg, cfg).dropna(subset=['y'])
results = fit_per_location(data, cfg)
rows = []
for name, r in results.items():
    if r['model'] is None:
        rows.append(dict(sensor=name, error=r.get('error'))); continue
    m, e = r['model'], r['eval']
    rows.append(dict(sensor=name, NOC=m.n_components, n_train=m.n_train,
                     train_R2=round(m.train_r2,3), test_pts=e['n_testpoints'],
                     MAPE=round(e['mape'],1), corr=round(e['corr'],3)))
pd.DataFrame(rows)
